# Integration Demonstration Notebook

This notebook demonstrates the preprocessing integration flow:

```text
raw video / HLS playlist
        ↓
video ingestion and preprocessing pipeline
        ↓
processed frames, metadata, quality report, scene report, validation report
        ↓
model-ready RGB frame folders for downstream CV tasks
```

The notebook is intentionally focused on the preprocessing stage. It does **not** run YOLO or MMPose inside the notebook. After this notebook generates the processed RGB frame folders, those folders can be used as input to downstream tasks such as object detection or pose estimation.

## 1. User Settings

Update `INPUT_PATH` before running the notebook.

Examples:

```python
INPUT_PATH = REPO_ROOT / "dataset" / "sample.mp4"
INPUT_PATH = REPO_ROOT / "dataset" / "hls" / "playlist.m3u8"
INPUT_PATH = Path(r"C:\path\to\playlist.m3u8")
```

In [ ]:
from pathlib import Path
import json
import sys
import subprocess
from typing import Optional, Iterable

import pandas as pd
from IPython.display import display, Markdown, Image as IPyImage

try:
    from PIL import Image
except Exception:
    Image = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None


# ---------------------------------------------------------
# Resolve repository root
# ---------------------------------------------------------
# Run this notebook either from:
#   1. repository root, or
#   2. notebooks/ folder
cwd = Path.cwd().resolve()

if cwd.name == "notebooks":
    REPO_ROOT = cwd.parent
else:
    REPO_ROOT = cwd


# ---------------------------------------------------------
# User-editable paths
# ---------------------------------------------------------
CONFIG_PATH = REPO_ROOT / "configs" / "video_processing_analysis.yaml"

# Change this to your local video file, folder, or HLS playlist.
INPUT_PATH = REPO_ROOT / "dataset" / "sample.mp4"

OUTPUT_DIR = REPO_ROOT / "outputs"

# Set to False if the preprocessing output already exists
# and you only want to inspect previous results.
RUN_PREPROCESSING = True

# Set to "auto" to choose the latest generated output folder.
# Or manually set the name, for example: VIDEO_OUTPUT_NAME = "playlist"
VIDEO_OUTPUT_NAME = "auto"

print("Repository root :", REPO_ROOT)
print("Config path     :", CONFIG_PATH)
print("Input path      :", INPUT_PATH)
print("Output dir      :", OUTPUT_DIR)

## 2. Helper Functions

These helper functions are used for input checking, report loading, frame listing, and image display.

In [ ]:
def is_url_like(path_or_url) -> bool:
    value = str(path_or_url)
    return value.startswith("http://") or value.startswith("https://")


def check_inputs():
    if not CONFIG_PATH.exists():
        raise FileNotFoundError(f"Configuration file not found: {CONFIG_PATH}")

    if not is_url_like(INPUT_PATH):
        input_path = Path(INPUT_PATH)
        if not input_path.exists():
            raise FileNotFoundError(
                f"Input path not found: {input_path}\n"
                "Please update INPUT_PATH in Section 1."
            )


def run_command(cmd, cwd=None):
    print("Running command:")
    print(" ".join(str(x) for x in cmd))

    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        capture_output=True,
        text=True
    )

    if result.stdout:
        print("\nSTDOUT:\n")
        print(result.stdout)

    if result.stderr:
        print("\nSTDERR:\n")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")

    return result


def read_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def list_images(folder: Path, exts=(".jpg", ".jpeg", ".png", ".bmp", ".webp")):
    if not folder.exists():
        return []

    files = []
    for ext in exts:
        files.extend(folder.rglob(f"*{ext}"))
    return sorted(files)


def show_image_grid(paths, title=None, max_images=8, cols=4, figsize=(14, 6)):
    paths = list(paths)[:max_images]

    if len(paths) == 0:
        print("No images found.")
        return

    if Image is None or plt is None:
        for p in paths:
            display(IPyImage(filename=str(p)))
        return

    rows = (len(paths) + cols - 1) // cols
    fig = plt.figure(figsize=figsize)

    if title:
        fig.suptitle(title, fontsize=14)

    for i, path in enumerate(paths):
        ax = fig.add_subplot(rows, cols, i + 1)
        img = Image.open(path).convert("RGB")
        ax.imshow(img)
        ax.set_title(path.name, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


def find_video_output_dirs(output_dir: Path):
    if not output_dir.exists():
        return []

    candidates = []
    for item in output_dir.iterdir():
        if not item.is_dir():
            continue

        if (item / "manifests").exists() or (item / "frames").exists():
            candidates.append(item)

    return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)


def select_video_output_dir() -> Path:
    if VIDEO_OUTPUT_NAME != "auto":
        selected = OUTPUT_DIR / VIDEO_OUTPUT_NAME
        if not selected.exists():
            raise FileNotFoundError(f"Selected output folder does not exist: {selected}")
        return selected

    candidates = find_video_output_dirs(OUTPUT_DIR)
    if not candidates:
        raise RuntimeError(
            f"No processed output folder found inside {OUTPUT_DIR}. "
            "Run the preprocessing step first."
        )

    return candidates[0]


def count_segment_frames(video_output_dir: Path) -> pd.DataFrame:
    good_dir = video_output_dir / "frames" / "good"
    rows = []

    if not good_dir.exists():
        return pd.DataFrame(rows)

    for segment_dir in sorted(good_dir.glob("segment_*")):
        rgb_dir = segment_dir / "rgb"
        gray_dir = segment_dir / "gray"

        rows.append({
            "segment": segment_dir.name,
            "rgb_frames": len(list_images(rgb_dir)),
            "gray_frames": len(list_images(gray_dir)),
            "rgb_path": str(rgb_dir.relative_to(REPO_ROOT)) if rgb_dir.exists() else "",
            "gray_path": str(gray_dir.relative_to(REPO_ROOT)) if gray_dir.exists() else "",
        })

    return pd.DataFrame(rows)


def select_representative_rgb_segment(video_output_dir: Path) -> Optional[Path]:
    segment_df = count_segment_frames(video_output_dir)

    if segment_df.empty:
        return None

    # Select the segment with the largest number of RGB frames.
    segment_df = segment_df.sort_values("rgb_frames", ascending=False)
    segment_name = segment_df.iloc[0]["segment"]

    rgb_dir = video_output_dir / "frames" / "good" / segment_name / "rgb"
    return rgb_dir if rgb_dir.exists() else None

## 3. Run the Preprocessing Pipeline

This cell calls the repository's main pipeline script:

```text
src/video_processing_analysis_pipeline.py
```

It produces structured outputs under the `outputs/` directory.

In [ ]:
if RUN_PREPROCESSING:
    check_inputs()

    cmd = [
        sys.executable,
        REPO_ROOT / "src" / "video_processing_analysis_pipeline.py",
        "--config", CONFIG_PATH,
        "--input", INPUT_PATH,
        "--output", OUTPUT_DIR,
    ]

    run_command(cmd, cwd=REPO_ROOT)
else:
    print("RUN_PREPROCESSING is False. Skipping preprocessing run.")

## 4. Select the Generated Output Folder

The pipeline creates one output folder per input video or stream. This cell selects the latest generated output folder by default.

In [ ]:
video_output_dir = select_video_output_dir()

print("Selected output folder:")
print(video_output_dir)
print("\nRelative path:")
print(video_output_dir.relative_to(REPO_ROOT))

print("\nTop-level contents:")
for item in sorted(video_output_dir.iterdir()):
    print(" -", item.name)

## 5. Load Metadata and Reports

This section loads the main output reports generated by the preprocessing pipeline.

In [ ]:
metadata_path = video_output_dir / "embedded_metadata" / "metadata.json"
manifest_path = video_output_dir / "manifests" / "manifest.json"
frames_manifest_path = video_output_dir / "manifests" / "frames_manifest.jsonl"
quality_report_path = video_output_dir / "manifests" / "quality_report.json"
scene_report_path = video_output_dir / "manifests" / "scene_report.json"
validation_report_path = video_output_dir / "manifests" / "validation_report.json"

report_paths = {
    "metadata": metadata_path,
    "manifest": manifest_path,
    "frames_manifest": frames_manifest_path,
    "quality_report": quality_report_path,
    "scene_report": scene_report_path,
    "validation_report": validation_report_path,
}

for name, path in report_paths.items():
    status = "FOUND" if path.exists() else "missing"
    print(f"{name:18s} {status:8s} {path}")

In [ ]:
# Display compact JSON report summaries.
for report_name, report_path in report_paths.items():
    if report_name == "frames_manifest":
        continue

    if not report_path.exists():
        continue

    display(Markdown(f"### {report_name}"))

    try:
        data = read_json(report_path)
        rows = []
        for key, value in data.items():
            if isinstance(value, (dict, list)):
                value = json.dumps(value)[:600]
            rows.append({"key": key, "value": value})
        display(pd.DataFrame(rows))
    except Exception as e:
        print(f"Could not read {report_path}: {e}")

## 6. Load the Frame Manifest

The frame manifest is the most important integration artifact. It stores one record per exported frame, including paths, timestamps, quality information, and segment information.

In [ ]:
if frames_manifest_path.exists():
    frames_df = read_jsonl(frames_manifest_path)
    print("Number of frame-manifest rows:", len(frames_df))
    display(frames_df.head(15))
else:
    frames_df = pd.DataFrame()
    print("frames_manifest.jsonl not found.")

## 7. Scene Segment Summary

Good frames are organized by detected segment. This table shows how many RGB and grayscale frames are available in each segment.

In [ ]:
segment_df = count_segment_frames(video_output_dir)

if segment_df.empty:
    print("No segment folders found under frames/good/.")
else:
    display(segment_df)

## 8. Display Sample Good RGB Frames

These are model-ready RGB frames. Downstream CV models can use these folders directly.

In [ ]:
rgb_segment_dir = select_representative_rgb_segment(video_output_dir)

if rgb_segment_dir is None:
    print("No RGB segment folder found.")
else:
    print("Selected RGB segment folder:")
    print(rgb_segment_dir.relative_to(REPO_ROOT))

    rgb_images = list_images(rgb_segment_dir)
    print("Number of RGB frames in selected segment:", len(rgb_images))

    show_image_grid(
        rgb_images,
        title="Sample Good RGB Frames",
        max_images=8,
        cols=4,
        figsize=(14, 6)
    )

## 9. Display Grayscale Frames If Available

Grayscale frames are useful for low-level analysis such as blur, brightness, and contrast.

In [ ]:
if rgb_segment_dir is None:
    print("No selected segment.")
else:
    gray_segment_dir = rgb_segment_dir.parent / "gray"
    gray_images = list_images(gray_segment_dir)

    print("Selected grayscale folder:")
    print(gray_segment_dir.relative_to(REPO_ROOT) if gray_segment_dir.exists() else gray_segment_dir)
    print("Number of grayscale frames:", len(gray_images))

    show_image_grid(
        gray_images,
        title="Sample Grayscale Frames",
        max_images=8,
        cols=4,
        figsize=(14, 6)
    )

## 10. Display Poor-Quality Frames

Poor-quality frames are separated from good frames according to the configured quality thresholds.

In [ ]:
poor_dir = video_output_dir / "frames" / "poor_quality"
poor_images = list_images(poor_dir)

print("Poor-quality folder:")
print(poor_dir.relative_to(REPO_ROOT) if poor_dir.exists() else poor_dir)
print("Number of poor-quality frames:", len(poor_images))

show_image_grid(
    poor_images,
    title="Poor-Quality Frame Examples",
    max_images=6,
    cols=3,
    figsize=(12, 6)
)

## 11. Display Saved Analysis Images If Available

If the pipeline saved analysis images such as quality curves or scene-boundary plots, this cell displays them.

In [ ]:
analysis_dir = video_output_dir / "analysis"
preview_dir = video_output_dir / "previews"

analysis_images = list_images(analysis_dir)
preview_images = list_images(preview_dir)

selected_plots = []
for path in analysis_images + preview_images:
    name = path.name.lower()
    if any(token in name for token in ["quality", "scene", "shot", "boundary", "segment", "plot"]):
        selected_plots.append(path)

if not selected_plots:
    print("No analysis plot images found.")
    print("Use the JSON reports and frame folders for integration verification.")
else:
    for path in selected_plots[:10]:
        print(path.relative_to(REPO_ROOT))
        display(IPyImage(filename=str(path)))

## 12. Output Paths for Downstream CV Tasks

The preprocessing stage is complete when model-ready RGB frames and reports are generated.

Use the RGB segment folder as input to downstream CV tasks such as object detection or pose estimation.

In [ ]:
if rgb_segment_dir is not None:
    print("Recommended downstream CV input folder:")
    print(rgb_segment_dir)

    print("\nRelative path:")
    print(rgb_segment_dir.relative_to(REPO_ROOT))

print("\nUseful generated artifacts:")

useful_paths = [
    video_output_dir / "frames" / "good",
    video_output_dir / "frames" / "poor_quality",
    frames_manifest_path,
    quality_report_path,
    scene_report_path,
    validation_report_path,
]

for path in useful_paths:
    print(f"{'FOUND' if path.exists() else 'missing':8s} {path.relative_to(REPO_ROOT) if path.exists() else path}")

## 13. Integration Demonstration Summary

This notebook demonstrates the required integration flow:

```text
raw video / HLS input
        ↓
preprocessing pipeline
        ↓
processed frames and reports
        ↓
model-ready RGB frame folders
```

The most important downstream input is:

```text
outputs/<video_id>/frames/good/segment_xxx/rgb/
```

These RGB frame folders can be used as input for downstream computer vision tasks such as:

- object detection using YOLO
- human pose estimation using MMPose
- tracking
- action recognition
- sports analytics